# Stage 2: small, very clean demo subset (silver -> gold)

**Purpose:** a tiny, trustworthy dataset for a **demo / proof of concept** (a chatbot or a report on top of it). It is **not** representative of the full data and must not be used for accuracy or prevalence statistics.

**Approach:** every row must be clean *from the raw Excel file itself*: all quality problems found during exploration are filtered out, and every rule is counted in a log. Then a small, balanced sample per rat is drawn. Column names and codes are made human-readable so a chatbot can use them directly.

Input: `../data/silver/data_stage1.parquet` | Output: `../data/gold/apopo_demo_subset.(parquet|csv)` and `apopo_demo_data_dictionary.csv`

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

SILVER = Path("../data/silver/data_stage1.parquet")
GOLD = Path("../data/gold")
GOLD.mkdir(parents=True, exist_ok=True)

SEED = 42
POS_PER_RAT = 80    # clinic-positive rows per rat (positives are rare, so they are over-sampled)
NEG_PER_RAT = 250   # clinic-negative rows per rat
MAX_SNIFF_MS = 10_000

df = pd.read_parquet(SILVER)
print(df.shape)

(204740, 27)


## 1. Quality filters (with a log)

| # | Rule | Why |
|---|---|---|
| 1 | Sample type is `BigNewPot` or `Old pot` | removes surrogate markers, unknown and rare types |
| 2 | Evaluation threshold > 0 | HIT can only be verified when the threshold is known |
| 3 | 0 < SniffTime <= 10 s | removes zero (missing or no sniff) and extreme values |
| 4 | HIT equals `SniffTime >= threshold` | removes rows where the recorded HIT contradicts the rule |
| 5 | Temperature > 0 | 0 is a missing-value code |
| 6 | Session start/end present and duration 1-60 min | removes broken timestamps |
| 7 | Session is operational (remarks start with "Discrimination" or "Sputum", no "Surrogate" or "TEST") | removes training and surrogate-mixed sessions |
| 8 | Sample has no microscopy / clinic GeneXpert conflict (checked on all rows) | keeps the reference unambiguous |
| 9 | Sample is not a "new case" (clinic negative, APOPO positive; checked on all rows) | its true status is ambiguous |
| 10 | Not a known positive with a clinic-negative result | contradictory label |

In [2]:
# sample-level exclusions are decided on the FULL data, so all presentations of an ambiguous sample are removed
new_case = (df["ID_BL_DOTS"] == 1) & df["ID_BL_APOPO"].notna() & (df["ID_BL_APOPO"] != 1)
new_case_samples = set(df.loc[new_case, "ID_SAMPLE"])
gxp = df["ID_GXP_DOTS"]
conflict = ((df["ID_BL_DOTS"] > 1) & (gxp == 1)) | ((df["ID_BL_DOTS"] == 1) & (gxp > 1))
conflict_samples = set(df.loc[conflict, "ID_SAMPLE"])

log = [("start (silver)", len(df))]


def apply(name, mask):
    global df
    df = df[mask].copy()
    log.append((name, len(df)))


apply("1 sample type BigNewPot / Old pot", df["SAMPLE_TYPE"].isin(["BigNewPot", "Old pot"]))
apply("2 evaluation threshold > 0", df["tblEVALUATION_SniffThreshold"] > 0)
apply("3 0 < SniffTime <= 10 s", df["SniffTime"].between(1, MAX_SNIFF_MS))
apply("4 HIT consistent with threshold", (df["SniffTime"] >= df["tblEVALUATION_SniffThreshold"]) == df["HIT"].astype(bool))
apply("5 temperature > 0", df["TEMPERATURE"] > 0)

duration_min = (df["END_TIME"] - df["START_TIME"]) * 24 * 60   # Excel serials are in days
apply("6 valid session start/end (1-60 min)", duration_min.between(1, 60))

remarks = df["tblEVALUATION_SESSION_REMARKS"].fillna("")
apply("7 operational session (no training/surrogate)",
      remarks.str.match(r"^(Discrimination|Sputum)") & ~remarks.str.contains("Surrogate|TEST", case=False))

apply("8 sample has no microscopy / GeneXpert conflict", ~df["ID_SAMPLE"].isin(conflict_samples))
apply("9 sample is not a new case", ~df["ID_SAMPLE"].isin(new_case_samples))
apply("10 no known-positive with clinic-negative result", ~(df["STATUS_KNOWNPOS"] & (df["ID_BL_DOTS"] == 1)))

log_df = pd.DataFrame(log, columns=["step", "rows_remaining"])
log_df["rows_removed"] = -log_df["rows_remaining"].diff().fillna(0).astype(int)
log_df

,step,rows_remaining,rows_removed
0,start (silver),204740,0
1,1 sample type BigNewPot / Old pot,172338,32402
2,2 evaluation threshold > 0,34607,137731
3,3 0 < SniffTime <= 10 s,32278,2329
4,4 HIT consistent with threshold,31904,374
5,5 temperature > 0,26146,5758
6,6 valid session start/end (1-60 min),25785,361
7,7 operational session (no training/surrogate),14847,10938
8,8 sample has no microscopy / GeneXpert conflict,14847,0
9,9 sample is not a new case,14847,0


## 2. Make it readable: dates, names, labels, outcome

In [3]:
def to_dt(s):
    return pd.to_datetime(s, unit="D", origin="1899-12-30").dt.round("s")


def bacterial_label(code):
    if code == 1:
        return "Negative"
    if 2 <= code <= 10:
        return f"{code - 1} AFB"
    if 11 <= code <= 13:
        return f"{code - 10}+"
    if 15 <= code <= 24:
        return f"{code - 5} AFB"
    return "Unknown"


def pseudonym(patient_id):
    return "P" + hashlib.sha256(f"apopo-poc-{patient_id}".encode()).hexdigest()[:8]


gxp_label = df["ID_GXP_DOTS"].map(lambda v: "Not tested" if pd.isna(v) else ("Negative" if v == 1 else "Positive"))
apopo_label = df["ID_BL_APOPO"].map(lambda v: "Not tested" if pd.isna(v) else ("Negative" if v == 1 else "Positive"))
positive = df["ID_BL_DOTS"] > 1
indicated = df["HIT"].astype(bool)

clean = pd.DataFrame({
    "session_id": df["ID_EVALUATION_SESSION"],
    "session_date": to_dt(df["SESSION_DATE"]).dt.date,
    "session_start": to_dt(df["START_TIME"]),
    "session_end": to_dt(df["END_TIME"]),
    "session_type": df["tblEVALUATION_SESSION_REMARKS"].str.strip(),
    "configuration": df["CONFIGURATION_NAME"].str.title(),
    "temperature_c": df["TEMPERATURE"],
    "rat": df["RAT_NAME"],
    "sample_id": df["ID_SAMPLE"],
    "patient_ref": df["ID_PATIENT"].map(pseudonym),
    "clinic": df["DOTS_NAME"],
    "sample_type": df["SAMPLE_TYPE"],
    "times_presented": df["REUSED"],
    "run": df["RUN"],
    "hole": df["HOLE"],
    "position": df["RUN"] + df["HOLE"].astype(str),
    "sniff_time_ms": df["SniffTime"],
    "sniff_time_s": (df["SniffTime"] / 1000).round(2),
    "threshold_ms": df["tblEVALUATION_SniffThreshold"],
    "rat_indicated": indicated,
    "rewarded": df["REWARD"].astype(bool),
    "is_known_positive": df["STATUS_KNOWNPOS"].astype(bool),
    "is_blind_positive": df["STATUS_BLINDPOS"].astype(bool),
    "clinic_microscopy": df["ID_BL_DOTS"].map(bacterial_label),
    "clinic_genexpert": gxp_label,
    "apopo_confirmation": apopo_label,
    "reference_result": np.where(positive, "Positive", "Negative"),
    "outcome": np.select([indicated & positive, ~indicated & positive, indicated & ~positive],
                         ["True positive", "False negative", "False positive"], default="True negative"),
}).reset_index(drop=True)

clean.shape

(14847, 28)

## 3. Validate the clean set
`reference_result` is based on clinic microscopy only (`> 1` = Positive). `outcome` compares the rat's indication with that reference.

In [4]:
assert clean["sample_id"].notna().all()
assert clean.notna().all().all(), "unexpected nulls"
assert (clean["sniff_time_ms"] > 0).all() and (clean["sniff_time_ms"] <= MAX_SNIFF_MS).all()
assert (clean["threshold_ms"] > 0).all()
assert (clean["rat_indicated"] == (clean["sniff_time_ms"] >= clean["threshold_ms"])).all()
assert (~clean["rewarded"] | clean["rat_indicated"]).all()          # reward never without an indication
assert (clean["session_end"] > clean["session_start"]).all()
assert clean.groupby("sample_id")["clinic_microscopy"].nunique().max() == 1
print("clean set passes all checks:", len(clean), "rows")
print(clean["outcome"].value_counts())

clean set passes all checks: 14847 rows
outcome
True negative     12037
True positive      1388
False positive     1115
False negative      307
Name: count, dtype: int64


## 4. Draw the small demo subset
A fixed-seed sample per rat: up to `POS_PER_RAT` clinic-positive and `NEG_PER_RAT` clinic-negative rows. Positives are **over-sampled on purpose** so a demo has enough of each outcome. Therefore the subset's positive share and accuracy are **not** the real values.

In [5]:
parts = []
for (rat, ref), g in clean.groupby(["rat", "reference_result"]):
    n = POS_PER_RAT if ref == "Positive" else NEG_PER_RAT
    parts.append(g.sample(n=min(n, len(g)), random_state=SEED))

demo = pd.concat(parts).sort_values(["session_start", "position"]).reset_index(drop=True)
demo.insert(0, "row_id", np.arange(1, len(demo) + 1))

print(f"demo subset: {len(demo):,} rows, {demo['rat'].nunique()} rats, "
      f"{demo['sample_id'].nunique():,} samples, {demo['session_id'].nunique()} sessions")
demo.groupby("rat").size().to_frame("rows").T

demo subset: 4,222 rows, 14 rats, 1,563 samples, 220 sessions


rat,Bieber,Chamy,Chilleta,Ella,Gea,Kenenisa,Malaika,Mayele,Orpheus,Salvina,Splinter,Tamasha,Tirunesh,Tivane
rows,275,330,245,267,330,330,330,330,135,330,330,330,330,330


In [6]:
demo["outcome"].value_counts().to_frame("rows")

,rows
outcome,
True negative,2860
True positive,800
False positive,383
False negative,179


In [7]:
demo.head(10)

,row_id,session_id,session_date,session_start,session_end,session_type,configuration,temperature_c,rat,sample_id,...,threshold_ms,rat_indicated,rewarded,is_known_positive,is_blind_positive,clinic_microscopy,clinic_genexpert,apopo_confirmation,reference_result,outcome
0,1,22032,2023-09-04,2023-09-04 10:35:00,2023-09-04 10:44:51,Discrimination 5 Holes,Line E-5H,21.0,Tamasha,476623,...,1500,True,True,True,False,2+,Not tested,Not tested,Positive,True positive
1,2,22046,2023-09-06,2023-09-06 13:10:22,2023-09-06 13:19:08,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Tivane,476623,...,1500,False,False,True,False,2+,Not tested,Not tested,Positive,False negative
2,3,22059,2023-09-07,2023-09-07 14:54:12,2023-09-07 15:02:30,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Splinter,672667,...,2000,False,False,True,False,1+,Not tested,Positive,Positive,False negative
3,4,22059,2023-09-07,2023-09-07 15:04:27,2023-09-07 15:13:44,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Tivane,672416,...,2000,True,True,False,False,5 AFB,Not tested,Not tested,Positive,True positive
4,5,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672421,...,1250,True,False,False,False,Negative,Not tested,Not tested,Negative,False positive
5,6,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672633,...,1250,True,True,False,False,2+,Not tested,Not tested,Positive,True positive
6,7,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672420,...,1250,True,False,False,False,Negative,Not tested,Not tested,Negative,False positive
7,8,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672412,...,1250,True,False,False,False,Negative,Negative,Not tested,Negative,False positive
8,9,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672404,...,1250,True,False,False,False,Negative,Negative,Not tested,Negative,False positive
9,10,22060,2023-09-08,2023-09-08 13:39:17,2023-09-08 13:57:48,Discrimination 5 Holes 30 Samples,Line E-5H,21.0,Malaika,672422,...,1250,False,False,False,False,Negative,Not tested,Not tested,Negative,True negative


## 5. Data dictionary and export

In [8]:
dictionary = pd.DataFrame([
    ("row_id", "Row number in this demo file"),
    ("session_id", "Evaluation-session identifier (several rats and rows share it)"),
    ("session_date", "Date of the evaluation session"),
    ("session_start", "Evaluation start (not an individual sniff timestamp)"),
    ("session_end", "Evaluation end"),
    ("session_type", "Session description from the remarks, e.g. 'Discrimination 10 Holes 10 Bars'"),
    ("configuration", "Cage setup, e.g. Line C"),
    ("temperature_c", "Room temperature in degrees Celsius"),
    ("rat", "Rat evaluating the sample"),
    ("sample_id", "Sample identifier; a sample can be presented many times, also to several rats"),
    ("patient_ref", "Pseudonymised patient reference (not the original patient id)"),
    ("clinic", "Partner clinic that supplied the sample"),
    ("sample_type", "Sample container type (BigNewPot or Old pot)"),
    ("times_presented", "Number of times this sample has been presented across sessions"),
    ("run", "Sample bar / presentation block letter"),
    ("hole", "Position within the run"),
    ("position", "Run letter + hole number, e.g. A1"),
    ("sniff_time_ms", "Recorded sniff duration in milliseconds"),
    ("sniff_time_s", "Recorded sniff duration in seconds"),
    ("threshold_ms", "Sniff-time threshold for this evaluation; a sniff at or above it is an indication"),
    ("rat_indicated", "True = the rat indicated the sample as suspect (a screening signal, not a diagnosis)"),
    ("rewarded", "True = a reward was recorded (a hit does not always give a reward)"),
    ("is_known_positive", "Sample presented as a known positive (control)"),
    ("is_blind_positive", "Clinic-positive sample presented as negative to avoid handler cues"),
    ("clinic_microscopy", "Partner clinic microscopy result: Negative, 1-19 AFB, or 1+/2+/3+ (categories, not concentrations)"),
    ("clinic_genexpert", "Partner clinic GeneXpert result: Negative, Positive or Not tested"),
    ("apopo_confirmation", "APOPO lab confirmation: Negative, Positive or Not tested (not tested is NOT negative)"),
    ("reference_result", "Positive if clinic microscopy is positive, else Negative"),
    ("outcome", "Rat indication vs. reference_result: True positive, False negative, False positive, True negative"),
], columns=["column", "description"])

assert set(dictionary["column"]) == set(demo.columns), set(demo.columns) ^ set(dictionary["column"])

demo.to_parquet(GOLD / "apopo_demo_subset.parquet", index=False)
demo.to_csv(GOLD / "apopo_demo_subset.csv", index=False)
dictionary.to_csv(GOLD / "apopo_demo_data_dictionary.csv", index=False)
log_df.to_csv(GOLD / "apopo_demo_cleaning_log.csv", index=False)
print("written to", GOLD.resolve())
dictionary

written to C:\Users\SebbeBlancke\Documents\Projects\hackton_apopo\data\gold


,column,description
0,row_id,Row number in this demo file
1,session_id,Evaluation-session identifier (several rats an...
2,session_date,Date of the evaluation session
3,session_start,Evaluation start (not an individual sniff time...
4,session_end,Evaluation end
5,session_type,"Session description from the remarks, e.g. 'Di..."
6,configuration,"Cage setup, e.g. Line C"
7,temperature_c,Room temperature in degrees Celsius
8,rat,Rat evaluating the sample
9,sample_id,Sample identifier; a sample can be presented m...


## Notes and limitations
- This is a POC subset: heavily filtered (about 7% of the rows survive the quality rules) and re-balanced. Do not report accuracy, sensitivity or prevalence from it.
- Quality rules removed all rows with an unknown threshold, so the subset only represents sessions where the threshold was recorded.
- `reference_result` uses clinic microscopy only. `rat_indicated` is a screening signal, not a diagnosis.
- The pseudonymised `patient_ref` is a salted hash; there is no mapping back to the original ID in the gold files.